# AJNR AI Publication-Trends Pipeline

Driver notebook. Each section runs one cached stage; see `README.md` for setup.

**Before running:** export `SCOPUS_API_KEY` (run from an institutional IP that
subscribes to Scopus), and launch the local vLLM server (`LLM_BASE_URL`
defaults to `http://localhost:8000/v1`).

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option('display.max_colwidth', 80)

from ajnr_ai_trends import config, acquire, enrich, normalize, extract, taxonomy
from ajnr_ai_trends import embed, cluster, trends, influence, viz, report
from ajnr_ai_trends.llm_client import LLMClient
from ajnr_ai_trends.scopus import ScopusClient

cfg = config.CONFIG
print('Data dir:', cfg.tables_dir.parent)

## 0. Health checks: Scopus entitlement + LLM endpoint

In [ ]:
# Confirm the LLM server is reachable
LLMClient(cfg).health()

In [ ]:
# Confirm your API key unlocks the FULL Abstract Retrieval view.
# Grab one EID from a quick search, then probe it.
sc = ScopusClient(cfg)
probe_hits = sc.search(acquire.build_query(cfg), count=5, max_results=5)
probe_sid = (probe_hits[0]['dc:identifier'] or '').replace('SCOPUS_ID:', '')
print('Probing scopus_id', probe_sid)
sc.probe_entitlement(probe_sid)

## 1-3. Acquire → enrich → normalize

In [ ]:
candidates = acquire.acquire_corpus(cfg)
records = enrich.enrich_corpus(candidates, cfg)
tables = normalize.normalize(records, cfg)
papers = tables['papers']
print(papers.shape)
papers[['year','title','citedby_count','n_authors','n_references']].head()

## 4-5. LLM extraction + subfield taxonomy

In [ ]:
papers = extract.extract_papers(papers, cfg)
papers = taxonomy.assign_subfields(papers, cfg, use_llm=True)
# tighten corpus to LLM-confirmed AI papers
papers = papers[papers['is_ai_relevant'].fillna(True)].copy()

subfields_long = (papers[['eid','year','citedby_count','citations_per_year','subfields']]
                  .explode('subfields').dropna(subset=['subfields'])
                  .rename(columns={'subfields':'subfield'}))
print('AI papers:', len(papers), '| subfield assignments:', len(subfields_long))
papers[['year','title','clinical_task','model_family','subfields']].head()

## 6-7. Embeddings + topic discovery

In [ ]:
emb = embed.embed_papers(papers, cfg)
topic_summary = cluster.discover_topics(papers, emb, cfg, label_with_llm=True)
papers_topics = pd.read_parquet(cfg.tables_dir / 'papers_topics.parquet')
topic_summary[['label','n_papers','total_citations','top_terms']]

## 8. Trends

In [ ]:
tt = trends.compute_all(papers, subfields_long, cfg)
display(tt['papers_per_year'])
display(tt['subfield_growth'])
display(tt['keyword_bursts'].head(10))
display(tt['top_cited_per_year'])

## 9. Influence

In [ ]:
inf = influence.compute_all(tables, subfields_long, cfg)
display(inf['author_influence'].head(15))
display(inf['affil_influence'].head(15))

## Figures

In [ ]:
from IPython.display import Image, display
figs = [
    viz.plot_volume_impact(tt['papers_per_year'], cfg),
    viz.plot_subfield_stream(tt['subfield_year_matrix'], cfg),
    viz.plot_subfield_heatmap(tt['subfield_share_matrix'], cfg),
    viz.plot_growth(tt['subfield_growth'], cfg),
    viz.plot_hype_vs_impact(tt['hype_vs_impact'], cfg),
    viz.plot_topic_scatter(papers_topics, cfg),
    viz.plot_country_trends(inf['country_trends'], cfg=cfg),
    viz.plot_coauthor_network(inf['_graph'], inf['coauthor_centrality'], cfg=cfg),
]
for f in figs:
    display(Image(filename=str(f)))

## 10. LLM narrative report

In [ ]:
from IPython.display import Markdown
path = report.narrative_report(papers, subfields_long, tt, inf, cfg)
Markdown(path.read_text())

---
### One-shot alternative
```python
from ajnr_ai_trends import pipeline
out = pipeline.run_all()
```